<a href="https://colab.research.google.com/github/v35001/aoid-team2-ahiyp/blob/main/code/09_data_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
# 1. 데이터 불러오기
try:
    df = pd.read_spss("/content/HN24_ALL.sav")
except ImportError:
    import os
    os.system('pip install pyreadstat')
    df = pd.read_spss("/content/HN24_ALL.sav")

print(f"Step 0. 2024년 KNHANES 원본 데이터: N = {len(df):,}")

# 실제 확인된 변수명을 적용하여 분석용 매핑을 정의합니다.
age_col = 'age'       # 나이
dm_col = 'DM4_dg'    # 당뇨병 의사 진단 여부 (기존 HE_DM 대체)
hp_col = 'HE_HP'     # 고혈압 유병 여부
ob_col = 'HE_obe'    # 비만 여부 (기존 HE_OB 대체)
incm_col = 'incm'    # 소득 수준
edu_col = 'edu'      # 교육 수준
bs_col = 'BS3_1'     # 흡연 여부
bd_col = 'BD1_11'    # 음주 여부

print(f"\n[변수 매핑 완료]")
print(f"- 나이 변수: {age_col}")
print(f"- 당뇨(노출): {dm_col}")
print(f"- 고혈압(결과): {hp_col}")
print(f"- 비만(공변량): {ob_col}")

# 2. 포함 기준 (만 19세 이상)
df[age_col] = pd.to_numeric(df[age_col], errors='coerce')
# ★ 수정 포인트: 끝에 .copy()를 붙여서 독립된 데이터로 만듦
df_step1 = df[df[age_col] >= 19].copy()
print(f"\nStep 1. 만 19세 이상 성인 포함: N = {len(df_step1):,} (제외: {len(df) - len(df_step1):,}명)")

# 3. 노출변수(당뇨병) 결측 및 무응답 제외
# 일반적으로 8(모름), 9(무응답)을 결측으로 간주
df_step1[dm_col] = pd.to_numeric(df_step1[dm_col], errors='coerce')
# ★ 수정 포인트: 끝에 .copy() 추가
df_step2 = df_step1[~df_step1[dm_col].isin([8, 9]) & df_step1[dm_col].notnull()].copy()
print(f"Step 2. 노출변수({dm_col}) 결측 제외: N = {len(df_step2):,} (제외: {len(df_step1) - len(df_step2):,}명)")

# 4. 결과변수(고혈압) 결측 및 무응답 제외
df_step2[hp_col] = pd.to_numeric(df_step2[hp_col], errors='coerce')
# ★ 수정 포인트: 끝에 .copy() 추가
df_step3 = df_step2[~df_step2[hp_col].isin([8, 9]) & df_step2[hp_col].notnull()].copy()
print(f"Step 3. 결과변수({hp_col}) 결측 제외: N = {len(df_step3):,} (제외: {len(df_step2) - len(df_step3):,}명)")

# 5. 주요 공변량 결측 및 무응답 제외
covariates = [ob_col, incm_col, edu_col, bs_col, bd_col]

# 공변량 변수들을 숫자형으로 안전하게 변환
for col in covariates:
    df_step3[col] = pd.to_numeric(df_step3[col], errors='coerce')

df_final = df_step3.dropna(subset=covariates)
print(f"Step 4. 공변량 결측 제외: N = {len(df_final):,} (제외: {len(df_step3) - len(df_final):,}명)")

# ★ 마지막 줄 오류 수정
print("-" * 50)
print(f">> 최종 분석 대상자 수: N = {len(df_final):,}명")

Step 0. 2024년 KNHANES 원본 데이터: N = 6,997

[변수 매핑 완료]
- 나이 변수: age
- 당뇨(노출): DM4_dg
- 고혈압(결과): HE_HP
- 비만(공변량): HE_obe

Step 1. 만 19세 이상 성인 포함: N = 6,033 (제외: 964명)
Step 2. 노출변수(DM4_dg) 결측 제외: N = 5,491 (제외: 542명)
Step 3. 결과변수(HE_HP) 결측 제외: N = 5,458 (제외: 33명)
Step 4. 공변량 결측 제외: N = 5,300 (제외: 158명)
--------------------------------------------------
>> 최종 분석 대상자 수: N = 5,300명
